# Late-tail persistence atlas

This notebook is the slower companion to `reports/late-tail-persistence-atlas.md`.
It reads the tile CSV, rebuilds the per-power summaries, and checks which center halos survive the harder read.


In [ ]:
import csv
from collections import defaultdict
from pathlib import Path

rows = []
with (Path('..') / 'art' / 'late-tail-persistence-atlas.csv').open() as handle:
    for row in csv.DictReader(handle):
        parsed = {key: float(value) if key not in {'power', 'budget', 'threshold', 'tile_x', 'tile_y'} else int(value) for key, value in row.items()}
        rows.append(parsed)
len(rows)


## Rebuild the per-power summaries

The report is comparing the old scouting read against a harder ultra-late read. This cell recreates those numbers directly from the tile table.


In [ ]:
grouped = defaultdict(list)
for row in rows:
    grouped[(row['power'], row['budget'])].append(row)

def summarize(tile_rows):
    total = sum(row['sample_count'] for row in tile_rows)
    grid_late = sum(row['sample_count'] * row['late_fraction'] for row in tile_rows) / total
    center_rows = sorted(tile_rows, key=lambda row: abs((row['x_min'] + row['x_max']) / 2.0) + abs((row['y_min'] + row['y_max']) / 2.0))[:4]
    center_late = sum(row['late_fraction'] for row in center_rows) / len(center_rows)
    unresolved = sum(row['sample_count'] * row['unresolved_fraction'] for row in tile_rows) / total
    return grid_late, center_late, unresolved

summary = {}
for power in [3, 6, 9, 12]:
    summary[(power, 40)] = summarize(grouped[(power, 40)])
    summary[(power, 80)] = summarize(grouped[(power, 80)])
summary


## Rank the powers by surviving ultra-late mass

If the new sidecar is honest, the higher-power center halos should survive the harder read much better than the lower-power ones.


In [ ]:
sorted([(power, summary[(power, 80)][0], summary[(power, 80)][1]) for power in [3, 6, 9, 12]], key=lambda item: item[1], reverse=True)
